# Prompt format: plain `Question:/Answer:` vs chat template

Holds the model fixed and varies **only** the prompt format, so any FSR/NLL gap
is a *pure format effect*. This turns reviewer LvWk's "prompt mismatch" from an
asserted confound into a measured one.

Per format we report:
- **gold-answer NLL** — how accessible the implanted fact is under that format,
- **greedy-contains FSR proxy** — does greedy decoding surface the ground truth.

The pre-unlearn **elicitation gap** = `plain_NLL − chat_NLL`. If plain
under-elicits the fact *before any unlearning*, then forgetting scored under
plain prompts is partly a format artifact. Set `ADAPTER_OVERRIDE` to an unlearned
adapter to also compare pre vs post and test whether forgetting **transfers
across formats** (a robustness signal). Run once per `MODEL`.


In [1]:
!pip install -q transformers peft datasets accelerate

In [11]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## Setup — HF login (Kaggle T4x2)


In [6]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")   # Kaggle > Add-ons > Secrets
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token)
print("GPUs:", torch.cuda.device_count())   # T4x2 -> 2; device_map='auto' shards the model


GPUs: 2


## Config


In [30]:
import json, numpy as np

MODEL = "gemma"          # "llama" | "gemma"

CFG = {
    "llama": dict(base="meta-llama/Llama-3.2-3B",
                  adapter="Novaspree/factify-3B-adapter", dtype=torch.float16),
    "gemma": dict(base="google/gemma-3-4b-it",
                  adapter="Novaspree/factify-Gemma3-adapter-1", dtype=torch.bfloat16),
}[MODEL]

ADAPTER_OVERRIDE = None      # set to unlearned-adapter path for post-unlearn comparison
N_SAMPLES = 100
SEED = 42
OUT_DIR = "results/analysis"
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)
print("MODEL:", MODEL, "| adapter:", ADAPTER_OVERRIDE or CFG["adapter"])


MODEL: gemma | adapter: Novaspree/factify-Gemma3-adapter-1


## Data + model


In [31]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

HF_DATASET = "Novaspree/factify_5K_enriched"
def _load(files):
    for f in files:
        try: return load_dataset(HF_DATASET, data_files=f, split="train")
        except Exception: pass
    raise RuntimeError(files)
forget_ds = _load(["forget/forget_set_fixed.json", "forget_set_fixed.json"])
idx = np.random.RandomState(SEED).permutation(len(forget_ds))[:N_SAMPLES]
forget = [forget_ds[int(i)] for i in idx]
print("forget samples:", len(forget))

tok = AutoTokenizer.from_pretrained(CFG["base"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(CFG["base"], torch_dtype=CFG["dtype"], device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_OVERRIDE or CFG["adapter"], is_trainable=False)
model.eval()
DEV = next(model.parameters()).device
print("loaded on", DEV)


forget samples: 100


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

loaded on cuda:0


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.10.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.10.self_attn.k_proj.lora_B.default.weight',

## Two prompt builders + metrics


In [32]:
def prompt_plain(q):
    return f"Question: {q}\nAnswer:"

def prompt_chat(q):
    return tok.apply_chat_template([{"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True)

def encode_fmt(q, a, fmt):
    if fmt == "plain":
        prompt = prompt_plain(q); full = prompt + " " + a; add_special = True
    else:
        prompt = prompt_chat(q); full = prompt + a; add_special = False
    plen = tok(prompt, return_tensors="pt", add_special_tokens=add_special)["input_ids"].shape[1]
    enc = tok(full, return_tensors="pt", truncation=True, max_length=512,
              add_special_tokens=add_special).to(DEV)
    labels = enc["input_ids"].clone(); labels[:, :plen] = -100
    return enc, labels, prompt

@torch.no_grad()
def gold_nll(q, a, fmt):
    enc, labels, _ = encode_fmt(q, a, fmt)
    logits = model(**enc).logits.float()[0][:-1]
    tgt = enc["input_ids"][0][1:]; mask = labels[0][1:] != -100
    if mask.sum() == 0: return None
    lp = torch.log_softmax(logits[mask], dim=-1)
    return -lp[range(int(mask.sum())), tgt[mask]].mean().item()

@torch.no_grad()
def greedy_contains(q, a, fmt):
    _, _, prompt = encode_fmt(q, a, fmt)
    add_special = fmt == "plain"
    inp = tok(prompt, return_tensors="pt", add_special_tokens=add_special).to(DEV)
    eot = tok.convert_tokens_to_ids("<|eot_id|>")
    eos = [i for i in {tok.eos_token_id, eot} if i is not None and i >= 0]
    out = model.generate(**inp, max_new_tokens=64, do_sample=False,
                         eos_token_id=eos, pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return a.strip().lower() in gen.lower()


## Run: plain vs chat


In [33]:
def format_prompt_eval(question):
    return (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )

def build_prompt(question, fmt):
    if fmt == "chat":
        return format_prompt_eval(question)
    elif fmt == "plain":
        return question
    else:
        raise ValueError(f"unknown fmt: {fmt}")

In [34]:
import json
import numpy as np
import torch
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Prompt formatting (manual Llama-3 template, no apply_chat_template)
# ---------------------------------------------------------------------
def format_prompt_eval(question):
    return (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )

def build_prompt(question, fmt):
    if fmt == "chat":
        return format_prompt_eval(question)
    elif fmt == "plain":
        return question
    else:
        raise ValueError(f"unknown fmt: {fmt}")


# ---------------------------------------------------------------------
# Core eval functions
# ---------------------------------------------------------------------
@torch.no_grad()
def gold_nll(question, answer, fmt):
    """Mean per-token negative log-likelihood of `answer` given the prompt."""
    prompt = build_prompt(question, fmt)

    # "chat" prompt already contains literal <|begin_of_text|>, so don't
    # let the tokenizer add another BOS on top of it.
    add_special = False if fmt == "chat" else True

    prompt_ids = tok(prompt, return_tensors="pt",
                     add_special_tokens=add_special).input_ids.to(DEV)
    full_text = prompt + answer
    full_ids = tok(full_text, return_tensors="pt",
                   add_special_tokens=add_special).input_ids.to(DEV)

    if full_ids.shape[1] <= prompt_ids.shape[1]:
        return None  # answer contributed no tokens, skip

    labels = full_ids.clone()
    labels[:, :prompt_ids.shape[1]] = -100  # mask out prompt tokens

    out = model(full_ids, labels=labels)
    return out.loss.item()


@torch.no_grad()
def greedy_contains(question, answer, fmt, max_new_tokens=64):
    """Greedy-decode a continuation and check if it contains the gold answer."""
    prompt = build_prompt(question, fmt)
    add_special = False if fmt == "chat" else True

    inputs = tok(prompt, return_tensors="pt",
                add_special_tokens=add_special).to(DEV)

    gen_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    gen_text = tok.decode(
        gen_ids[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    return answer.strip().lower() in gen_text.strip().lower()


# ---------------------------------------------------------------------
# Main eval loop
# ---------------------------------------------------------------------
res = {}
for fmt in ("plain", "chat"):
    nlls, hits = [], []
    for s in tqdm(forget, desc=fmt):
        v = gold_nll(s["question"], s["answer"], fmt)
        if v is not None:
            nlls.append(v)
        hits.append(greedy_contains(s["question"], s["answer"], fmt))
    res[fmt] = {
        "gold_nll": float(np.mean(nlls)),
        "contains_rate": float(np.mean(hits)),
    }

report = {
    "model": CFG["base"],
    "adapter": ADAPTER_OVERRIDE or CFG["adapter"],
    "by_format": res,
    "elicitation_gap_nll": res["plain"]["gold_nll"] - res["chat"]["gold_nll"],
}

print(json.dumps(report, indent=2))
print(f"\n{'format':<8}{'gold_nll':>12}{'contains':>12}")
for f in ("plain", "chat"):
    print(f"{f:<8}{res[f]['gold_nll']:>12.3f}{res[f]['contains_rate']:>12.1%}")

print(
    "\nelicitation_gap_nll > 0 => plain under-elicits the fact; forgetting scored"
    " under plain prompts is partly a format artifact, not knowledge removal."
)

path = f"{OUT_DIR}/plain_vs_chat_{CFG['base'].split('/')[-1]}.json"
json.dump(report, open(path, "w"), indent=2)
print("saved ->", path)

plain:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


chat:   0%|          | 0/100 [00:00<?, ?it/s]

{
  "model": "google/gemma-3-4b-it",
  "adapter": "Novaspree/factify-Gemma3-adapter-1",
  "by_format": {
    "plain": {
      "gold_nll": 5.6417253366112705,
      "contains_rate": 0.18
    },
    "chat": {
      "gold_nll": 9.527369581460952,
      "contains_rate": 0.03
    }
  },
  "elicitation_gap_nll": -3.885644244849682
}

format      gold_nll    contains
plain          5.642       18.0%
chat           9.527        3.0%

elicitation_gap_nll > 0 => plain under-elicits the fact; forgetting scored under plain prompts is partly a format artifact, not knowledge removal.
saved -> results/analysis/plain_vs_chat_gemma-3-4b-it.json


## Reading the result

- **Large positive elicitation gap** → the earlier plain-format baselines
  (GA/GA+KL/RO-FT before the fix) were under-querying the adapter; report the
  chat-format numbers as the valid comparison and cite this gap as the reason.
- **Small gap** → the prompt mismatch didn't materially change conclusions; say
  so, with this number as evidence. Either outcome is a controlled answer to LvWk.
- **Pre vs post (via `ADAPTER_OVERRIDE`)** → if forgetting under chat does *not*
  reduce the fact under plain (or vice versa), the unlearning is format-specific,
  i.e. shallow — pair with paraphrase/relearning tests.
